In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT / OUTPUT
# =============================================================================

INPUT_CSV = Path("data") / "MASTER_VARIABLES.csv"
OUTPUT_CSV = Path("data") / "exposure.csv"

df = pd.read_csv(INPUT_CSV)

print("Input shape:", df.shape)

# =============================================================================
# Z-SCORE + BINNING FUNCTIONS
# =============================================================================

def zscore(x):
    std = x.std(ddof=0)
    if std == 0:
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std


def classify(z):
    if z <= -1.5:
        return 1
    elif z <= -0.5:
        return 2
    elif z <= 0.5:
        return 3
    elif z <= 1.5:
        return 4
    else:
        return 5


# =============================================================================
# 1. DISTRICT-MONTH POPULATION AGGREGATION
# =============================================================================
# Exposure driver = sum_population (block → district)

district_df = (
    df.groupby(["district", "timeperiod"], as_index=False)
      .agg(
          sum_population=("sum_population", "sum")
      )
)

# =============================================================================
# 2. MONTH-WISE Z-SCORE (ACROSS DISTRICTS)
# =============================================================================

district_df["pop_z"] = (
    district_df.groupby("timeperiod")["sum_population"]
    .transform(zscore)
)

# =============================================================================
# 3. BINNING INTO EXPOSURE CLASS (1–5)
# =============================================================================

district_df["exposure"] = district_df["pop_z"].apply(classify)

# =============================================================================
# 4. SAVE OUTPUT
# =============================================================================

district_df.to_csv(OUTPUT_CSV, index=False)

print("Saved:", OUTPUT_CSV)

# =============================================================================
# 5. CHECKS
# =============================================================================

print("\nPreview:")
print(district_df.head())

print("\nExposure distribution:")
print(district_df["exposure"].value_counts().sort_index())

Input shape: (7222, 20)
Saved: data/exposure.csv

Preview:
  district timeperiod  sum_population     pop_z  exposure
0   Anugul    2023_01    1.436473e+06 -0.244147         3
1   Anugul    2023_02    1.436473e+06 -0.244147         3
2   Anugul    2023_03    1.436473e+06 -0.244147         3
3   Anugul    2023_04    1.436473e+06 -0.244147         3
4   Anugul    2023_05    1.436473e+06 -0.244147         3

Exposure distribution:
exposure
2    230
3    253
4    138
5     69
Name: count, dtype: int64
